## Running the storm time series code

In [ ]:
import bisect
import pathlib
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from barrier3d.load_input import _guess_format
from distfit import distfit

In [ ]:
def yearly_storms(
    datadir=".",
    storm_list_name="StormList_20k_VCR_Berm1pt9m_Slope0pt04.csv",  # can by .py or .csv
    mean_yearly_storms=8.3,
    SD_yearly_storms=5.9,
    MHW=0.46,  # m NAVD88
    StormStart=2,
    BermEl=1.9,  # m NAVD88, just used for plotting
    model_years=10000,
    bPlot=True,
    bSave=False,
    savedir=".",
    output_filename="StormList_10kyrs_VCR_Berm1pt9m_Slope0pt04",
):
    """
    Use a normal distribution (provided the mean and standard deviation)
    to select a random number of storms per year from a list of
    multivariate sea storms, created using the Wahl et al., 2016 method.
    """

    datadir = pathlib.Path(datadir)

    # convert to decameters
    MHW = MHW / 10
    BermEl = BermEl / 10 - MHW  # just for plotting

    # load list of storms (created using multivariateSeaStorm.m)
    fmt = _guess_format(datadir / storm_list_name)
    if fmt == "npy":
        StormList = np.load(datadir / storm_list_name, allow_pickle=True)
    elif fmt == "csv":
        StormList = np.loadtxt(
            datadir / storm_list_name, delimiter=",", encoding="utf-8-sig"
        )

    # pad with zeros until storms start
    StormSeries = np.zeros([StormStart, 5])

    for t in range(StormStart, model_years):
        # Calculate number of storms in year
        numstorm = round(np.random.normal(mean_yearly_storms, SD_yearly_storms))
        if numstorm < 0:
            numstorm = 0
        stormTS = np.zeros([numstorm, 5])

        # Select storms for year
        for n in range(numstorm):
            storm = random.randint(1, len(StormList) - 1)

            dur = StormList[storm, 1]  # Duration
            Rhigh = StormList[storm, 2]  # TWL
            period = StormList[storm, 4]  # Tp
            Rlow = StormList[storm, 6]  # Rlow

            stormTS[n, 0] = t
            stormTS[n, 1] = Rhigh / 10 - MHW  # make relative to MHW
            stormTS[n, 2] = Rlow / 10 - MHW
            stormTS[n, 3] = period
            stormTS[n, 4] = round(
                dur / 2
            )  # Divided by two assuming TWL only for only half of storm
            # (NOTE from KA: we could probably do better here, like assume that the
            # TWL follows a distribution; or create a time series as in the Wahl 2019
            # follow up paper for MSSM. Future work!)

        StormSeries = np.vstack([StormSeries, stormTS])

    if bPlot:
        # Plots
        Bin = np.linspace(-1, 4.6, 57)

        plt.figure()
        surgetidecalc = StormSeries[:, 2] * 10
        plt.hist(surgetidecalc, bins=Bin)
        plt.title("StormSeries Rlow")
        plt.xlabel("m MHW")

        plt.figure()
        twlcalc = StormSeries[:, 1] * 10
        plt.hist(twlcalc, bins=Bin)
        plt.title("StormSeries Rhigh")
        plt.xlabel("m MHW")

        plt.figure()
        fig = plt.gcf()
        fig.set_size_inches(16, 4)
        plt.plot(twlcalc)
        plt.plot(
            np.arange(0, len(twlcalc), 1),
            np.ones(len(twlcalc)) * BermEl * 10,
            "r--",
        )
        plt.xlabel("Storm")
        plt.ylabel("TWL (m MHW)")
        plt.legend(["TWL", "BermEl"])

        print("Max TWL (m MHW): ", max(StormSeries[:, 1]) * 10)
        print("Max Rexcess (m above berm): ", (max(StormSeries[:, 1]) - BermEl) * 10)
        print(
            "% Rhigh > BermEl: ", (twlcalc > (BermEl * 10)).sum() / len(twlcalc) * 100
        )
        print(
            "% Rlow  > BermEl: ",
            (surgetidecalc > (BermEl * 10)).sum() / len(surgetidecalc) * 100,
        )

    if bSave:
        np.save(savedir + output_filename, StormSeries)

    return StormSeries

In [ ]:
# datadir = "./data"
datadir = (
    "C:/Users/Lexi/PycharmProjects/CASCADE/notebooks/synthetic_storm_creation_NCB/data/"
)

for storm_n in range(1, 101):
    yearly_storms(
        datadir=datadir,
        storm_list_name="StormList_20k_NCB_Berm1pt46m_Slope0pt03_t-student.csv",  # this is == "cascade_default_storm_list.csv"
        mean_yearly_storms=25.4,
        SD_yearly_storms=5.3,
        MHW=0.36,  # m NAVD88
        StormStart=2,
        BermEl=1.46,  # m NAVD88, just used for plotting
        model_years=101,
        bPlot=True,
        bSave=True,
        output_filename="StormSeries_100yrs_inclusive_NCB_Berm1pt46m_Slope0pt03_{}".format(
            storm_n
        ),
        savedir="C:/Users/Lexi/PycharmProjects/CASCADE/cascade/data/outwash_data/storms/slope0pt03/",
    )

    yearly_storms(
        datadir=datadir,
        storm_list_name="StormList_20k_NCB_Berm1pt46m_Slope0pt002_t-student.csv",  # this is == "cascade_default_storm_list.csv"
        mean_yearly_storms=13.2,
        SD_yearly_storms=3.8,
        MHW=0.36,  # m NAVD88
        StormStart=2,
        BermEl=1.46,  # m NAVD88, just used for plotting
        model_years=101,
        bPlot=True,
        bSave=True,
        output_filename="StormSeries_100yrs_inclusive_NCB_Berm1pt46m_Slope0pt002_{}".format(
            storm_n
        ),
        savedir="C:/Users/Lexi/PycharmProjects/CASCADE/cascade/data/outwash_data/storms/slope0pt002/",
    )